<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.en/cap04/cap04.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Practical Part with Programming Exercises**


This list transforms the concepts from Chapter 4 into a practical track on segmentation and mathematical morphology. The programming exercises begin with thresholding and advance to labeling and component descriptors, always using small matrices so that each pixel can be verified by hand.

> ### ❗ Common rule for morphological programming exercises
>
> In neighborhood operations, **do not apply padding**. For each pixel, evaluate only the positions of the structuring element that fall within the image domain. This is the same idea behind the didactic implementations in `morph.py`, such as `mm.dil0`, `mm.ero0`, `mm.dil1`, and `mm.label0`: the neighborhood is clipped by the valid domain of the image.

### 🎯 Objective of this Notebook

The notebook allows you to develop, validate, organize, and test solutions for **Programming Exercises (EPs)** in interactive environments, such as Colab, using the same test cases as Moodle, and copying them there only when recording the official grade.

#### *Download*

Download `morph.py` and `testsuite.py` by running the cell below:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Executing Tests
To evaluate the tests, run `TestSuite("EP04_01.extension").run()` in a new cell, replacing the extension with that of the language used (`.py`, `.java`, `.c`, `.cpp`, `.js`, or `.r`). The system downloads the test cases from GitHub, runs the program, and computes the grade automatically.

To test Python code directly, without saving a file, use `run_code(code)` by passing the code as a *string* in a variable `code`:

```python
code = """
from morph import mm
# ... your code here ...
"""
TestSuite("EP04_01").run_code(code)
```

### EP04_01 🎚️ Global Thresholding with a Fixed Threshold

In **document scanners** and **barcode reading systems**, the first processing step is always to separate what is "object" (ink, text, bars) from what is "background" (paper, packaging). **Global thresholding** does exactly this: it compares each pixel to a single threshold $T$ and decides, in real time, whether it belongs to the light class or the dark class. It is the simplest segmentation operator—and yet, it underlies a large portion of industrial visual inspection *pipelines*.
See [Figure 4.1](#fig-04-sim-ep0401-limiar) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Threshold:** Read the integer $T$ (decision threshold).
3. **Data:** Read the integer values of the original matrix row by row.
4. **Mapping:** For each pixel $p$, compute the new value using the equation:

$$
p' =
\begin{cases}
255, & \text{if } p > T \\
0, & \text{if } p \le T
\end{cases}
$$
5. **Output:** Display the binarized matrix with dimensions $L \times C$.

#### 📌 Computational Constraints

* **Binarization:** The output contains **only** the values $0$ or $255$.
* **Strict comparison:** The criterion uses $> T$ (pixels equal to $T$ become background).
* **Type:** The final result must be an integer.
* **Note:** This EP follows the OpenCV convention (`cv2.THRESL_BINARY`): only pixels with value **greater than** $T$ become white (`255`); pixels with value **equal to** $T$ remain black (`0`).

#### 🧠 Theoretical Foundation

| Parameter | Type | Visual Impact |
|-----------|------|----------------|
| **$T$ small** | Integer | Most pixels become white |
| **$T$ large**  | Integer | Most pixels become black |
| **$T$ well-chosen** | Integer | Clearly separates object and background |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $T$.
* Following lines: Integer elements of the original matrix.

**Output:**

* Binarized matrix with $L$ rows and $C$ columns, values $0$ or $255$ separated by spaces.

#### 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 2<br>4<br>100<br>0 99 100 180<br>255 30 120 80 | 0 0 0 255<br>255 0 255 0 | $T=100$: only pixels with value greater than 100 become white; <br>therefore, 99 and 100 become black. |
| 1<br>3<br>0<br>0 50 255 | 0 255 255 | $T=0$: only pixels with value strictly greater than 0 become white. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0401-limiar" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎚️ Simulator EP04_01: Global Thresholding</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = (p > T) ? 255 : 0</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Click on a cell in the <b>Original Input</b> to darken the pixel (−30) and right-click to lighten (+30). Adjust the threshold T for binarization.</p>

    <!-- Controle do Limiar T -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">T (Threshold)</label>
        <span id="sim_ep0401_vl_t" style="font-family:monospace;font-size:12px;font-weight:700;color:#2980b9;">128</span>
      </div>
      <input type="range" id="sim_ep0401_sl_t" min="0" max="255" step="1" value="128" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado Binarizado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original Input (Clickable)</span>
        <div id="sim_ep0401_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
      </div>

      <!-- Resultado Binarizado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Binarized Result (p')</span>
        <div id="sim_ep0401_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Reset Threshold (T = 128)</button>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0401_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Formula applied: <b>(p > 128) ? 255 : 0</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0401(root){
    if (!root || root.dataset.simEp0401Init) return;
    root.dataset.simEp0401Init = "1";

    var slT      = root.querySelector('#sim_ep0401_sl_t');
    var vlT      = root.querySelector('#sim_ep0401_vl_t');
    var gridOrig = root.querySelector('#sim_ep0401_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0401_grid_new');
    var debugDiv = root.querySelector('#sim_ep0401_debug');

    var btnNew   = root.querySelector('#sim_ep0401_btnNew');
    var btnReset = root.querySelector('#sim_ep0401_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-30) | Botão direito: clareia (+30)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 30);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 30);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function render() {
      var T = parseInt(slT.value, 10) || 0;
      vlT.textContent = T;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>(p > ' + T + ') ? 255 : 0</b>';

      renderOrig();
      gridNew.innerHTML = '';

      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slT.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slT.value = '128';
      render();
    });

    render();
  }

  function tryInitSimEP0401(){
    var root = document.getElementById('sim-ep0401-limiar');
    if (root) initSimEP0401(root); else setTimeout(tryInitSimEP0401, 200);
  }
  tryInitSimEP0401();
})();
</script>
</div>
""")

**Figure 4.1:** EP04_01 Simulator: Global Thresholding by Fixed Threshold (p


In [ ]:
%%writefile EP04_01.py
# Python code

In [ ]:
TestSuite("EP04_01.py").run()

### EP04_02 📊 Otsu's Automatic Thresholding

Manually choosing the threshold $T$ works when illumination is stable, but in **digital microscopy** and **blood smear inspection**, each sample has different contrast — a fixed threshold would fail from image to image. **Otsu's method** solves this by autonomously finding the threshold that **maximizes the statistical separation** between the two pixel classes, making segmentation automatic and adaptive.
See [Figure 4.2](#fig-04-sim-ep0402-otsu) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Data:** Read the integer values of the original matrix row by row.
3. **Histogram:** Build the histogram $h[i]$, $i=0,\dots,255$, counting how many pixels have value $i$.
4. **Threshold search:** For each candidate $T$ from $1$ to $255$, compute the **between-class variance**:
$$
\sigma_B^2(T) = \frac{n_0 \cdot n_1}{N^2}\,(m_0 - m_1)^2
$$
where $n_0,n_1$ are the numbers of pixels with values $<T$ and $\geq T$, $m_0,m_1$ are their means, and $N=L\times C$.

5. **Selection:** The optimal threshold $T^*$ is the one that maximizes $\sigma_B^2(T)$ (in case of a tie, keep the **first** one found).
6. Application: Binarize the image using T*, applying:
$$
p' =
\begin{cases}
255, & \text{if } p > T^* \\
0, & \text{if } p \le T^*
\end{cases}
$$

#### 📌 Computational Constraints

* **Valid candidates:** Ignore $T$ that leaves $n_0=0$ or $n_1=0$ (empty class).
* **Tie-breaking:** Always keep the **first** $T$ that reached the maximum value of $\sigma_B^2$.
* **Type:** $T^*$ and the output matrix must be integers.
* **OpenCV convention:** The binarization follows `cv2.THRESL_BINARY`; pixels with value exactly equal to $T^*$ become black.


#### 🧠 Theoretical Background

| Concept | Meaning | Impact |
|----------|-------------|---------|
| **High $\sigma_B^2(T)$** | Classes well separated at $T$ | $T$ is a good threshold candidate |
| **Bimodal histogram** | Two distinct "peaks" | Otsu finds the valley between them |
| **Unimodal histogram** | A single "peak" | Otsu still chooses *some* $T$, but the segmentation is unreliable |


#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Following lines: Integer elements of the original matrix.

**Output:**

* Binarized matrix with $L$ rows and $C$ columns, values $0$ or $255$.

#### 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 4<br>4<br>12 12 12 200<br>12 12 200 200<br>12 200 200 200<br>200 200 200 200 | 0 0 0 255<br>0 0 255 255<br>0 255 255 255<br>255 255 255 255 | Clear bimodal histogram: 12 and 200 |
| 1<br>2<br>10 250 | 0 250 | Only two values: $T^*$ falls on the largest one |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0402-otsu" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 EP04_02 Simulator: Automatic Otsu</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">T* = argmax σ²_B(T)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Left click darkens (−25) and right click lightens (+25) the input pixels. Watch the optimal threshold T* adjust dynamically to the histogram.</p>

    <!-- Painel do Histograma e T* -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;">
      <div id="sim_ep0402_hist" style="display:flex;align-items:flex-end;gap:2px;height:100px;margin-bottom:8px;border-bottom:1px solid #e4dcc8;padding-bottom:2px;"></div>
      <p id="sim_ep0402_info" style="text-align:center;font-size:11.5px;font-family:monospace;font-weight:700;color:#26241d;margin:0;">T* = −</p>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Clicável vs Resultado Otsu -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original Input (Clickable)</span>
        <div id="sim_ep0402_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Otsu -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Otsu Result (p')</span>
        <div id="sim_ep0402_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botão de Nova Imagem -->
    <div style="text-align:center;">
      <button id="sim_ep0402_btnNew" style="padding:6px 14px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image (Two Groups)</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0402(root){
    if (!root || root.dataset.simEp0402Init) return;
    root.dataset.simEp0402Init = "1";

    var gridOrig = root.querySelector('#sim_ep0402_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0402_grid_new');
    var info     = root.querySelector('#sim_ep0402_info');
    var hist     = root.querySelector('#sim_ep0402_hist');
    var btnNew   = root.querySelector('#sim_ep0402_btnNew');

    var pixels = [];

    function generate() {
      var c1 = 20 + Math.floor(Math.random() * 40);
      var c2 = 180 + Math.floor(Math.random() * 60);
      pixels = [];
      for (var i = 0; i < 16; i++) {
        var base = (Math.random() < 0.5) ? c1 : c2;
        pixels.push(Math.max(0, Math.min(255, base + Math.floor(Math.random() * 16 - 8))));
      }
    }

    function otsu(pix) {
      var histArr = new Array(256).fill(0);
      pix.forEach(function(p){ histArr[p]++; });
      var N = pix.length, bestVar = -1, bestT = 0;
      var total = pix.reduce(function(a, b){ return a + b; }, 0);

      for (var T = 1; T < 256; T++) {
        var n0 = 0, s0 = 0;
        for (var i = 0; i < T; i++) {
          n0 += histArr[i];
          s0 += i * histArr[i];
        }
        var n1 = N - n0, s1 = total - s0;
        if (n0 === 0 || n1 === 0) continue;
        var m0 = s0 / n0, m1 = s1 / n1;
        var v = (n0 * n1) * (m0 - m1) * (m0 - m1) / (N * N);
        if (v > bestVar) {
          bestVar = v;
          bestT = T;
        }
      }
      return bestT;
    }

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-25) | Botão direito: clareia (+25)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 25);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 25);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function renderHist(T) {
      hist.innerHTML = '';
      var histArr = new Array(256).fill(0);
      pixels.forEach(function(p){ histArr[p]++; });
      var maxH = Math.max.apply(null, histArr);

      for (var i = 0; i < 256; i += 4) {
        var h = (histArr[i] / (maxH || 1)) * 100;
        var bar = document.createElement('div');
        var col = (i >= T) ? '#2980b9' : '#8a8371';
        bar.style.cssText = 'flex:1;height:' + h + '%;background:' + col + ';border-radius:2px 2px 0 0;';
        hist.appendChild(bar);
      }
    }

    function render() {
      var T = otsu(pixels);
      info.innerHTML = 'T* encontrado = <b>' + T + '</b>';
      renderOrig();
      renderHist(T);

      gridNew.innerHTML = '';
      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0402(){
    var root = document.getElementById('sim-ep0402-otsu');
    if (root) initSimEP0402(root); else setTimeout(tryInitSimEP0402, 200);
  }
  tryInitSimEP0402();
})();
</script>
</div>
""")

**Figure 4.2:** EP04_02 Simulator: Automatic Otsu Thresholding (T* = argmax σ²_B(T))


In [ ]:
%%writefile EP04_02.py
# Python code

In [ ]:
TestSuite("EP04_02.py").run()

### EP04_03 🌱 Flat Binary Dilation (mm.dil0)

In **particle microscopy** and in **OCR of worn license plates**, thin or discontinuous traces need to be "thickened" for recognition to work. **Morphological dilation** does exactly that: it expands bright regions using a structuring element $B$ — the same operation implemented in `morph.py` as `mm.dil0(f, B)`, used when $B$ is **flat** (without weights, only $0$/$1$).
See [Figure 4.3](#fig-04-sim-ep0403-dilatacao) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Image dimensions:** Read the integers $L$ (rows) and $C$ (columns) from $f$.
2. **Dimensions of $B$:** Read the integers $L_B$ (rows) and $C_B$ (columns) of the structuring element.
3. **Structuring element:** Read the matrix $B$ with values $0$ or $1$, row by row.
4. **Data:** Read the matrix $f$ (the original image), row by row.
5. **Reflection:** Construct $B_{ref}$, the version of $B$ reflected by $180°$ (rows and columns reversed) — exactly as `mm.dil0` does internally.
6. **Neighborhood without padding:** For each pixel $(y,x)$, traverse the positions $(by,bx)$ of $B_{ref}$ centered at $(y,x)$, using the offset
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Discard** every $(v_y,v_x)$ outside $[0,L)\times[0,C)$ — **do not pad with zeros**.
7. **Mapping:** Compute each output pixel as the **maximum** between $f(y,x)$ and all valid $f(v_y,v_x)$ whose corresponding position in $B_{ref}$ is $1$:
$$
g(y,x) = \max\Big(f(y,x),\ \max_{\substack{(v_y,v_x)\ \text{valid}\\ B_{ref}(by,bx)=1}} f(v_y,v_x)\Big)
$$
8. **Output:** Display the matrix $g$ with dimensions $L \times C$.

#### 📌 Computational Constraints

* **No padding:** Never invent neighbors outside the image; use only those that actually exist.
* **Mandatory reflection:** $B$ must be reflected before being applied (this is what distinguishes `mm.dil0` from a simple maximum search).
* **Edge robustness:** If no valid position of $B_{ref}=1$ falls within the domain for a given pixel, it **maintains its original value**.

#### 🧠 Theoretical Foundation

| Concept | Meaning | Visual Impact |
|----------|-------------|-----------------|
| **Dilation** | $g \geq f$ always (extensive) | Bright regions grow, dark holes shrink |
| **Larger $B$** | Wider neighborhood | More aggressive growth |
| **Reflection of $B$** | $B_{ref}(y,x) = B(-y,-x)$ | Ensures the formal Minkowski definition of dilation |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $L_B$.
* Line 4: Integer $C_B$.
* Next $L_B$ lines: integer elements ($0$ or $1$) of the matrix $B$.
* Next $L$ lines: integer elements of the matrix $f$.

**Output:**

* Matrix $g$ in $L$ rows and $C$ columns, integer values separated by spaces.

#### 📌 Examples

| Input | Output | Remark |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0<br>0 9 0<br>0 0 0 | 0 9 0<br>9 9 9<br>0 9 0 | Symmetric cross $B$: isolated point expands into a cross |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 200 200 200 80 | Horizontal $B$: each pixel "pulls" the maximum of row neighbors |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0403-dilatacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌱 Simulator EP04_03: Planar Dilation (mm.dil0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Toggle structuring element B (or select presets) and click cells of the original image f to light up or erase pixels.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Structuring Element B (Click to Toggle 0/1)</span>
      <div id="sim_ep0403_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0403_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Cross</button>
        <button id="sim_ep0403_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Box</button>
        <button id="sim_ep0403_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonal</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Dilatada g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original Image f (5×5)</span>
        <div id="sim_ep0403_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0403_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
      </div>

      <!-- Imagem Dilatada g -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Dilated g (f ⊕ B)</span>
        <div id="sim_ep0403_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0403_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = max over valid neighbors of reflected B
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0403(root){
    if (!root || root.dataset.simEp0403Init) return;
    root.dataset.simEp0403Init = "1";

    var gB      = root.querySelector('#sim_ep0403_grid_B');
    var gO      = root.querySelector('#sim_ep0403_grid_orig');
    var gN      = root.querySelector('#sim_ep0403_grid_new');
    var debugDiv= root.querySelector('#sim_ep0403_debug');

    var btnNew   = root.querySelector('#sim_ep0403_btnNew');
    var btnCross = root.querySelector('#sim_ep0403_btnCross');
    var btnBox   = root.querySelector('#sim_ep0403_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0403_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 0 : 1);
        }
        pixels.push(row);
      }
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) {
        out.push(M[i].slice().reverse());
      }
      return out;
    }

    function dilate(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var Bref = reflect(Bm);
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] > g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#16a085' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = dilate(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0403(){
    var root = document.getElementById('sim-ep0403-dilatacao');
    if (root) initSimEP0403(root); else setTimeout(tryInitSimEP0403, 200);
  }
  tryInitSimEP0403();
})();
</script>
</div>
""")

**Figure 4.3:** Simulator EP04_03: Binary Dilation (g = f ⊕ B)


In [ ]:
%%writefile EP04_03.py
# Python code

In [ ]:
TestSuite("EP04_03.py").run()

### EP04_04 🪨 Binary Erosion by Flat Structuring Element (mm.ero0)

If dilation thickens, **erosion** thins. In **cell counting systems**, it is used to **separate touching cells**: by "eating away" the borders of each region, thin connections between objects disappear even before any counting is performed. In `morph.py`, this is the operation `mm.ero0(f, B)` — the exact **dual** of dilation, and the only one of the two that does **not** reflect the structuring element.
See [Figure 4.4](#fig-04-sim-ep0404-erosao) for a simulation of this exercise.

#### 📋 Implementation Guidelines

1. **Image dimensions:** Read the integers $L$ (rows) and $C$ (columns) from $f$.
2. **Dimensions of $B$:** Read the integers $L_B$ (rows) and $C_B$ (columns) of the structuring element.
3. **Structuring element:** Read the matrix $B$ with values $0$ or $1$, row by row.
4. **Data:** Read the matrix $f$ (the original image), row by row.
5. **Neighborhood without padding (no reflection!):** For each pixel $(y,x)$, traverse the positions $(by,bx)$ of $B$ **in the original order** (without reflecting), using the same offset as in EP04_03:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$

**Discard** every $(v_y,v_x)$ outside $[0,L)\times[0,C)$.
6. **Mapping:** Compute each output pixel as the **minimum** between $f(y,x)$ and all valid $f(v_y,v_x)$ whose corresponding position in $B$ equals $1$:
$$
g(y,x) = \min\Big(f(y,x),\ \min_{\substack{(v_y,v_x)\ \text{valid}\\ B(by,bx)=1}} f(v_y,v_x)\Big)
$$
7. **Output:** Display the matrix $g$ with dimensions $L \times C$.

#### 📌 Computational Constraints

* **No reflection:** Unlike dilation, $B$ is used **exactly as read** — reflecting it here would be a serious conceptual error.
* **No padding:** Neighbors outside the image are simply ignored, never treated as $0$.
* **Edge robustness:** If no valid position of $B=1$ falls within the domain, the pixel retains its original value.

#### 🧠 Theoretical Background

| Concept | Meaning | Visual Impact |
|----------|-------------|-----------------|
| **Erosion** | $g \leq f$ always (anti-extensive) | Bright regions shrink, point noise disappears |
| **Duality** | $\text{ero}(f,B) = -\text{dil}(-f, B_{ref})$ | Erosion and dilation are mathematical "mirrors" |
| **Larger $B$** | More aggressive erosion | Thin objects disappear completely |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $L_B$.
* Line 4: Integer $C_B$.
* Next $L_B$ lines: integer elements ($0$ or $1$) of matrix $B$.
* Next $L$ lines: integer elements of matrix $f$.

**Output:**

* Matrix $g$ in $L$ rows and $C$ columns, integer values separated by spaces.

#### 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>9 9 9<br>9 0 9<br>9 9 9 | 9 0 9<br>0 0 0<br>9 0 9 | The central "hole" (0) propagates in a cross pattern |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 10 5 5 80 | Horizontal $B$: each pixel "pulls" the minimum of its row neighbors |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0404-erosao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪨 Simulator EP04_04: Planar Erosion (mm.ero0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Switch the structuring element B (or select the presets) and click the cells of the original image f to turn pixels on or off.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Structuring Element B (Click to Toggle 0/1)</span>
      <div id="sim_ep0404_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0404_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Cross</button>
        <button id="sim_ep0404_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Box</button>
        <button id="sim_ep0404_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonal</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Erodida g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original Image f (5×5)</span>
        <div id="sim_ep0404_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0404_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
      </div>

      <!-- Imagem Erodida g -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Eroded g (f ⊖ B)</span>
        <div id="sim_ep0404_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0404_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = min over valid neighbors of B (without reflection)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0404(root){
    if (!root || root.dataset.simEp0404Init) return;
    root.dataset.simEp0404Init = "1";

    var gB      = root.querySelector('#sim_ep0404_grid_B');
    var gO      = root.querySelector('#sim_ep0404_grid_orig');
    var gN      = root.querySelector('#sim_ep0404_grid_new');
    var debugDiv= root.querySelector('#sim_ep0404_debug');

    var btnNew   = root.querySelector('#sim_ep0404_btnNew');
    var btnCross = root.querySelector('#sim_ep0404_btnCross');
    var btnBox   = root.querySelector('#sim_ep0404_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0404_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 1 : 0);
        }
        pixels.push(row);
      }
    }

    function erode(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] < g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#c0392b' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = erode(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0404(){
    var root = document.getElementById('sim-ep0404-erosao');
    if (root) initSimEP0404(root); else setTimeout(tryInitSimEP0404, 200);
  }
  tryInitSimEP0404();
})();
</script>
</div>
""")

**Figure 4.4:** EP04_04 Simulator: Planar Binary Erosion (g = f ⊖ B)


In [ ]:
%%writefile EP04_04.py
# Python code

In [ ]:
TestSuite("EP04_04.py").run()

### EP04_05 🧹 Morphological Opening (Noise Removal)

Images captured by **low-cost sensors**, such as those on agricultural drones, often come dotted with small noise points—isolated pixels that represent nothing real. Applying erosion followed by dilation with the **same** structuring element produces the **opening**: it "cleans" points and thin protrusions, but returns the main object to nearly its original size. This is the classic combination used in **satellite image preprocessing** before any planted-area counting.
See [Figure 4.5](#fig-04-sim-ep0405-abertura) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Image dimensions:** Read integers $L$ (rows) and $C$ (columns) from $f$.
2. **Dimensions of $B$:** Read integers $L_B$ (rows) and $C_B$ (columns) of the structuring element.
3. **Structuring element:** Read matrix $B$ with values $0$ or $1$, row by row.
4. **Data:** Read binary matrix $f$ (values $0$ or $1$), row by row.
5. **Erosion:** Compute $e = f \ominus B$, using exactly the algorithm from EP04_04 (without reflecting $B$, without padding).
6. **Dilation:** Compute $g = e \oplus B$, using exactly the algorithm from EP04_03 (reflecting $B$, without padding)—but now applied to $e$, not to $f$.
7. **Output:** Display the resulting matrix $g$ (the **opening** of $f$ by $B$) with dimensions $L \times C$.

#### 📌 Computational Constraints

* **Fixed order:** It is **always** erosion first, then dilation—the reverse order defines another operator (closing, from the next EP).
* **Same $B$:** The structuring element used in erosion and dilation must be identical.
* **No padding in either step.**

#### 🧠 Theoretical Foundation

| Concept | Meaning | Visual Impact |
|----------|-------------|-----------------|
| **Anti-extensivity** | $g \subseteq f$ always | The opening never creates a new pixel, only removes |
| **Idempotence** | $\text{opening}(\text{opening}(f)) = \text{opening}(f)$ | Applying it again changes nothing further |
| **Isolated points** | Smaller than $B$ | Are completely eliminated |
| **Object core** | Larger than $B$ | Is recovered almost intact by the final dilation |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $L_B$.
* Line 4: Integer $C_B$.
* Next $L_B$ lines: integer elements ($0$ or $1$) of matrix $B$.
* Next $L$ lines: integer elements ($0$ or $1$) of matrix $f$.

**Output:**

* Resulting matrix with $L$ rows and $C$ columns, values $0$ or $1$.

#### 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 7<br>7<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0<br>0 1 0 0 0 1 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 1 0<br>0 0 0 0 0 0 0<br>0 1 0 0 0 0 1 | 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | Isolated points and the thin protrusion disappear; the central square survives |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0405-abertura" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧹 Simulator EP04_05: Morphological Opening</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊖ B) ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Click on the cells of <b>f original</b> to turn pixels on or off (create your own background noise!) and adjust the size of the structuring element B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#8e44ad;">Size of B (n×n box)</label><br>
      <input type="range" id="sim_ep0405_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#8e44ad;margin-top:6px;">
      <span id="sim_ep0405_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs e (Erosão) vs g (Abertura Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Clickable)</span>
        <div id="sim_ep0405_grid_f" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- e = f ⊖ B -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">e = f ⊖ B (Erosion)</span>
        <div id="sim_ep0405_grid_e" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = e ⊕ B -->
      <div style="background:#fafaf7;border:2px solid #8e44ad;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = e ⊕ B (Opening)</span>
        <div id="sim_ep0405_grid_g" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0405_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image (With Noise)</button>
      <button id="sim_ep0405_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Clear All</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0405(root){
    if (!root || root.dataset.simEp0405Init) return;
    root.dataset.simEp0405Init = "1";

    var slN     = root.querySelector('#sim_ep0405_sl_n');
    var vlN     = root.querySelector('#sim_ep0405_vl_n');
    var gF      = root.querySelector('#sim_ep0405_grid_f');
    var gE      = root.querySelector('#sim_ep0405_grid_e');
    var gG      = root.querySelector('#sim_ep0405_grid_g');
    var btnNew  = root.querySelector('#sim_ep0405_btnNew');
    var btnClear= root.querySelector('#sim_ep0405_btnClear');

    var L = 7, C = 7, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 2; y < 5; y++) {
        for (var x = 2; x < 5; x++) f[y][x] = 1;
      }
      for (var k = 0; k < 3; k++) {
        var ry = Math.floor(Math.random() * L), rx = Math.floor(Math.random() * C);
        if (f[ry][rx] === 0 && (ry < 1 || ry > 5 || rx < 1 || rx > 5)) f[ry][rx] = 1;
      }
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#8e44ad' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#8e44ad' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var e = erode(f, B);
      var g = dilate(e, B);

      paintEditable(gF, f);
      paintStatic(gE, e);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0405(){
    var root = document.getElementById('sim-ep0405-abertura');
    if (root) initSimEP0405(root); else setTimeout(tryInitSimEP0405, 200);
  }
  tryInitSimEP0405();
})();
</script>
</div>
""")

**Figure 4.5:** EP04_05 Simulator: Morphological Opening (g = (f ⊖ B) ⊕ B)


In [ ]:
%%writefile EP04_05.py
# Python Code

In [ ]:
TestSuite("EP04_05.py").run()

### EP04_06 🧩 Morphological Closing (Filling of Gaps)

In **fingerprint digitalization**, skin ridges sometimes become interrupted by dirt or dryness, creating small gaps in the continuous curve that should exist. **Closing** — dilation followed by erosion with the same structuring element — is the dual operator of opening: it **fills small holes and narrow indentations**, without significantly altering the external contour of the object. It is the standard step before extracting the skeleton of a fingerprint.
See [Figure 4.6](#fig-04-sim-ep0406-fechamento) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Image dimensions:** Read the integers $L$ (rows) and $C$ (columns) from $f$.
2. **Dimensions of $B$:** Read the integers $L_B$ (rows) and $C_B$ (columns) of the structuring element.
3. **Structuring element:** Read the matrix $B$ with values $0$ or $1$, row by row.
4. **Data:** Read the binary matrix $f$ (values $0$ or $1$), row by row.
5. **Dilation:** Compute $d = f \oplus B$, using exactly the algorithm from EP04_03 (reflecting $B$, without padding).
6. **Erosion:** Compute $g = d \ominus B$, using exactly the algorithm from EP04_04 (without reflecting $B$, without padding) — now applied to $d$, not to $f$.
7. **Output:** Display the resulting matrix $g$ (the **closing** of $f$ by $B$) with dimensions $L \times C$.

#### 📌 Computational Constraints

* **Fixed order:** It is **always** dilation first, then erosion — the reverse order corresponds to the opening from EP04_05.
* **Same $B$:** The structuring element used in the dilation and in the erosion must be identical.
* **No padding in either of the two stages.**

#### 🧠 Theoretical Foundation

| Concept | Meaning | Visual Impact |
|----------|-------------|-----------------|
| **Extensivity** | $g \supseteq f$ always | Closing never removes a pixel, only adds |
| **Idempotence** | $\text{close}(\text{close}(f)) = \text{close}(f)$ | Applying it again changes nothing further |
| **Small holes** | Smaller than $B$ | They are completely filled |
| **Duality** | $\text{close}(f) = \overline{\text{open}(\bar f)}$ | It is the opening applied to the "negative" of the image |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $L_B$.
* Line 4: Integer $C_B$.
* Next $L_B$ lines: integer elements ($0$ or $1$) of matrix $B$.
* Next $L$ lines: integer elements ($0$ or $1$) of matrix $f$.

**Output:**

* Resulting matrix in $L$ rows and $C$ columns, values $0$ or $1$.

#### 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 8<br>8<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 0 1 1 0 0<br>0 0 1 1 0 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 | 0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0 | The two non-adjacent internal holes are completely filled |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0406-fechamento" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧩 Simulator EP04_06: Morphological Closing</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊕ B) ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Click on cells of <b>f original</b> to turn pixels on or off (fill in internal holes!) and adjust the size of structuring element B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#2c7a7b;">Size of B (n×n Box)</label><br>
      <input type="range" id="sim_ep0406_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#2c7a7b;margin-top:6px;">
      <span id="sim_ep0406_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#2c7a7b;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs d (Dilatação) vs g (Fechamento Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(170px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Clickable)</span>
        <div id="sim_ep0406_grid_f" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- d = f ⊕ B -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">d = f ⊕ B (Dilation)</span>
        <div id="sim_ep0406_grid_d" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = d ⊖ B -->
      <div style="background:#fafaf7;border:2px solid #2c7a7b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2c7a7b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = d ⊖ B (Closing)</span>
        <div id="sim_ep0406_grid_g" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0406_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image (With Holes)</button>
      <button id="sim_ep0406_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Clear All</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0406(root){
    if (!root || root.dataset.simEp0406Init) return;
    root.dataset.simEp0406Init = "1";

    var slN     = root.querySelector('#sim_ep0406_sl_n');
    var vlN     = root.querySelector('#sim_ep0406_vl_n');
    var gF      = root.querySelector('#sim_ep0406_grid_f');
    var gD      = root.querySelector('#sim_ep0406_grid_d');
    var gG      = root.querySelector('#sim_ep0406_grid_g');
    var btnNew  = root.querySelector('#sim_ep0406_btnNew');
    var btnClear= root.querySelector('#sim_ep0406_btnClear');

    var L = 8, C = 8, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 1; y < 7; y++) {
        for (var x = 2; x < 6; x++) f[y][x] = 1;
      }
      f[3][3] = 0;
      f[4][4] = 0;
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#2c7a7b' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#2c7a7b' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var d = dilate(f, B);
      var g = erode(d, B);

      paintEditable(gF, f);
      paintStatic(gD, d);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0406(){
    var root = document.getElementById('sim-ep0406-fechamento');
    if (root) initSimEP0406(root); else setTimeout(tryInitSimEP0406, 200);
  }
  tryInitSimEP0406();
})();
</script>
</div>
""")

**Figure 4.6:** EP04_06 Simulator: Morphological Closing (g = (f ⊕ B) ⊖ B)


In [ ]:
%%writefile EP04_06.py
# Python code

In [ ]:
TestSuite("EP04_06.py").run()

### EP04_07 ⛰️ Weighted Dilation and Erosion (mm.dil1 / mm.ero1)

So far, the structuring element only indicated "this neighbor counts" or "does not count" — but in **digital elevation models** (used in GIS and urban drainage planning), each neighbor should have a **different weight** depending on the distance or the direction of the terrain. The **weighted** versions of dilation and erosion, implemented in `morph.py` as `mm.dil1(f, b)` and `mm.ero1(f, b)`, sum (or subtract) the weight of each neighbor before taking the maximum (or minimum) — generalizing everything done in the previous EPs.
See [Figure 4.7](#fig-04-sim-ep0407-pesos) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Image dimensions:** Read the integers $L$ (rows) and $C$ (columns) from $f$.
2. **Dimensions of $b$:** Read the integers $L_B$ (rows) and $C_B$ (columns) of the weighted structuring element.
3. **Weights:** Read the matrix $b$ of **integer** weights (which may be negative, zero, or positive), row by row.
4. **Data:** Read the matrix $f$ (the original image), row by row.
5. **Neighborhood without padding:** For each pixel $(y,x)$, iterate over **all** positions $(by,bx)$ of $b$ (not only where it would equal $1$ — here **every** weight participates), using the same offset as in previous EPs:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Discard** every $(v_y,v_x)$ outside $[0,L)\times[0,C)$.
6. **Weighted dilation:** Compute
$$
g_{dil}(y,x) = \max\Big(f(y,x),\ \max_{(v_y,v_x)\ \text{valid}} \big(f(v_y,v_x) + b(by,bx)\big)\Big)
$$
7. **Weighted erosion:** Compute, **using the same $b$ and without reflection**:
$$
g_{ero}(y,x) = \min\Big(f(y,x),\ \min_{(v_y,v_x)\ \text{valid}} \big(f(v_y,v_x) - b(by,bx)\big)\Big)
$$
8. **Output:** Display **first** the complete matrix $g_{dil}$, and **then** the complete matrix $g_{ero}$.

#### 📌 Computational Constraints

* **Neither reflects $b$** — the weighted version does not use reflection, even in dilation (unlike `mm.dil0`).
* **All weights participate:** There is no "$B=1$" filter here; even weight $0$ is included in the computation.
* **No padding:** neighbors outside the image are ignored, never virtually filled.
* **Type:** The output may contain negative values or values greater than $255$ — there is **no** *clipping* in this EP.
* **Hint:** To remove overflow messages when exceeding uint8 limits, include at the beginning of the code:
```python
import warnings
warnings.filterwarnings("ignore")
```

#### 🧠 Theoretical Foundation

| Concept | Meaning | Visual Impact |
|----------|-------------|-----------------|
| **Positive weight** | "Pulls" the neighbor's value upward in dilation | Simulates terrain rising in that direction |
| **Negative weight** | Reduces the neighbor's contribution | Simulates distance or directional attenuation |
| **Weighted duality** | $\text{ero1}(f,b) = -\text{dil1}(-f,b)$ | The symmetry between the two operations is maintained even with weights |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $L_B$.
* Line 4: Integer $C_B$.
* Next $L_B$ lines: integer elements (may be negative) of the matrix $b$.
* Next $L$ lines: integer elements of the matrix $f$.

**Output:**

* First, the matrix $g_{dil}$ in $L$ rows and $C$ columns.
* Then, the matrix $g_{ero}$ in $L$ rows and $C$ columns.

#### 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 2 1<br>0 1 0<br>10 20 30<br>40 50 60<br>70 80 90 | 50 60 61<br>80 90 91<br>81 91 92<br>8 9 19<br>9 10 20<br>39 40 50 | Central weight $2$ accelerates growth in dilation and shrinkage in erosion |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0407-pesos" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⛰️ Simulator EP04_07: Weights in the Structuring Element</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">dil1 / ero1</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Adjust the weights of the structuring element b with the sliders and observe the effect of dilation and erosion with weights on the matrix f.</p>

    <!-- Painel dos Pesos b -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Weights b (Adjust Sliders per Cell)</span>
      <div id="sim_ep0407_grid_b" style="display:grid;grid-template-columns:repeat(3, 70px);gap:8px;justify-content:center;user-select:none;"></div>
    </div>

    <!-- Comparativo em 3 Colunas: f original vs dil1 vs ero1 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:14px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original f</span>
        <div id="sim_ep0407_grid_f" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- dil1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">dil1(f, b) (Dilation)</span>
        <div id="sim_ep0407_grid_d" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ero1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">ero1(f, b) (Erosion)</span>
        <div id="sim_ep0407_grid_e" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0407(root){
    if (!root || root.dataset.simEp0407Init) return;
    root.dataset.simEp0407Init = "1";

    var gB = root.querySelector('#sim_ep0407_grid_b');
    var gF = root.querySelector('#sim_ep0407_grid_f');
    var gD = root.querySelector('#sim_ep0407_grid_d');
    var gE = root.querySelector('#sim_ep0407_grid_e');

    var b = [[0, 1, 0], [1, 2, 1], [0, 1, 0]];
    var f = [[10, 20, 30], [40, 50, 60], [70, 80, 90]];
    var L = 3, C = 3;

    function compute() {
      var oy = -3 / 2 + 0.5, ox = -3 / 2 + 0.5;
      var dil = f.map(function(r){ return r.slice(); });
      var ero = f.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < 3; by++) {
            for (var bx = 0; bx < 3; bx++) {
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                var cd = f[vy][vx] + b[by][bx];
                if (cd > dil[y][x]) dil[y][x] = cd;
                var ce = f[vy][vx] - b[by][bx];
                if (ce < ero[y][x]) ero[y][x] = ce;
              }
            }
          }
        }
      }
      return { dil: dil, ero: ero };
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(row, col){
            var wrap = document.createElement('div');
            wrap.style.cssText = 'display:flex;flex-direction:column;align-items:center;background:#fafaf7;border:1px solid #e4dcc8;border-radius:6px;padding:4px;box-sizing:border-box;';

            var val = document.createElement('div');
            val.style.cssText = 'font-family:monospace;font-weight:700;font-size:11px;color:#d35400;margin-bottom:2px;';
            val.textContent = b[row][col];

            var sl = document.createElement('input');
            sl.type = 'range';
            sl.min = '-5';
            sl.max = '5';
            sl.step = '1';
            sl.value = b[row][col];
            sl.style.cssText = 'width:56px;cursor:pointer;accent-color:#d35400;';

            sl.addEventListener('input', function(){
              b[row][col] = parseInt(sl.value, 10);
              val.textContent = b[row][col];
              renderAll();
            });

            wrap.appendChild(val);
            wrap.appendChild(sl);
            gB.appendChild(wrap);
          })(by, bx);
        }
      }
    }

    function paint(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = img[y][x];
          var inten = Math.min(255, Math.max(0, v));
          var fg = inten > 128 ? '#000000' : '#ffffff';
          c.style.cssText = 'width:52px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + inten + ',' + inten + ',' + inten + ');color:' + fg + ';box-sizing:border-box;';
          c.textContent = v;
          grid.appendChild(c);
        }
      }
    }

    function renderAll() {
      var res = compute();
      paint(gF, f);
      paint(gD, res.dil);
      paint(gE, res.ero);
    }

    renderB();
    renderAll();
  }

  function tryInitSimEP0407(){
    var root = document.getElementById('sim-ep0407-pesos');
    if (root) initSimEP0407(root); else setTimeout(tryInitSimEP0407, 200);
  }
  tryInitSimEP0407();
})();
</script>
</div>
""")

**Figure 4.7:** EP04_07 Simulator: Dilation and Erosion with Weights (mm.dil1 / mm.ero1)


In [ ]:
%%writefile EP04_07.py
# Python code

In [ ]:
TestSuite("EP04_07.py").run()

### EP04_08 🌋 Morphological Gradient, Top-hat, and Black-hat

In **automatic inspection of printed circuit boards**, three questions arise all the time: where are the **edges** of the components? Which **small bright details** (such as solder points) stand out from the background? What **dark recesses** (such as cracks) does the background conceal? A single erosion/dilation pair answers all three: the **morphological gradient** highlights contours, the **top-hat** reveals narrow peaks, and the **black-hat** reveals narrow valleys — three tools, one single neighborhood.
See [Figure 4.8](#fig-04-sim-ep0408-gradiente) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Image dimensions:** Read the integers $L$ (rows) and $C$ (columns) from $f$.
2. **Dimensions of $B$:** Read the integers $L_B$ (rows) and $C_B$ (columns) of the structuring element.
3. **Structuring element:** Read the matrix $B$ with values $0$ or $1$, row by row.
4. **Data:** Read the matrix $f$ (the original image, in grayscale), row by row.
5. **Basic operators:** Compute, exactly as in EPs 04_03 through 04_06:
   * $d = f \oplus B$ (dilation),
   * $e = f \ominus B$ (erosion),
   * $\text{opening} = e \oplus B$,
   * $\text{closing} = d \ominus B$.
6. **Morphological gradient:** $\text{grad}(y,x) = d(y,x) - e(y,x)$.
7. **Top-hat:** $\text{tophat}(y,x) = f(y,x) - \text{opening}(y,x)$.
8. **Black-hat:** $\text{blackhat}(y,x) = \text{closing}(y,x) - f(y,x)$.
9. **Output:** Display, **in this order**, the three complete matrices: gradient, top-hat, black-hat.

#### 📌 Computational Constraints

* **No padding at any intermediate stage** — dilation, erosion, opening, and closing follow the same neighborhood rules as in the previous EPs.
* **No clipping:** the three outputs may contain any integer value (the gradient is always $\geq 0$, but top-hat and black-hat may also be so).
* **Reuse:** $d$ and $e$ must be computed **only once** and reused to assemble opening, closing, and gradient.

#### 🧠 Theoretical Foundation

| Operator | Formula | What it reveals |
|----------|---------|----------------|
| **Gradient** | $d - e$ | Edges: zero in flat regions, high at transitions |
| **Top-hat** | $f - \text{opening}(f)$ | **Bright, thin** elements, smaller than $B$ |
| **Black-hat** | $\text{closing}(f) - f$ | **Dark, thin** elements, smaller than $B$ |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $L_B$.
* Line 4: Integer $C_B$.
* Next $L_B$ lines: integer elements ($0$ or $1$) of matrix $B$.
* Next $L$ lines: integer elements of matrix $f$.

**Output:**

* Gradient matrix with $L$ rows and $C$ columns.
* Top-hat matrix with $L$ rows and $C$ columns.
* Black-hat matrix with $L$ rows and $C$ columns.

#### 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 9<br>9<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 80 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 2 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10 | (gradient: $3\times3$ halo of $70$ around $(2,2)$ and $3\times3$ halo of $8$ around $(6,6)$, rest $0$)<br>(top-hat: single $70$ at $(2,2)$, rest $0$)<br>(black-hat: single $8$ at $(6,6)$, rest $0$) | Isolated peak becomes top-hat; isolated valley becomes black-hat; both appear in the gradient |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0408-gradiente" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌋 EP04_08 Simulator: Gradient / Top-hat / Black-hat</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">3 operators, 1 neighborhood</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Add peaks or valleys to matrix f and observe the simultaneous behavior of the gradient, top-hat, and black-hat operators.</p>

    <!-- Botões de Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0408_btn_pico" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">☀️ Add Peak</button>
      <button id="sim_ep0408_btn_vale" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">🕳️ Add Valley</button>
      <button id="sim_ep0408_btn_reset" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">↩ Clear All</button>
    </div>

    <!-- Comparativo em 4 Colunas: f, Gradiente, Top-hat, Black-hat -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(140px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Matriz f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">f (Input)</span>
        <div id="sim_ep0408_grid_f" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente -->
      <div style="background:#fafaf7;border:1px solid #8e44ad;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Gradient</span>
        <div id="sim_ep0408_grid_grad" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Top-hat -->
      <div style="background:#fafaf7;border:1px solid #d35400;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Top-hat</span>
        <div id="sim_ep0408_grid_th" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Black-hat -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Black-hat</span>
        <div id="sim_ep0408_grid_bh" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0408(root){
    if (!root || root.dataset.simEp0408Init) return;
    root.dataset.simEp0408Init = "1";

    var gF   = root.querySelector('#sim_ep0408_grid_f');
    var gGrad= root.querySelector('#sim_ep0408_grid_grad');
    var gTh  = root.querySelector('#sim_ep0408_grid_th');
    var gBh  = root.querySelector('#sim_ep0408_grid_bh');

    var btnPico  = root.querySelector('#sim_ep0408_btn_pico');
    var btnVale  = root.querySelector('#sim_ep0408_btn_vale');
    var btnReset = root.querySelector('#sim_ep0408_btn_reset');

    var L = 9, C = 9, f = [], B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];

    function resetMatrix() {
      f = Array.from({ length: L }, function(){ return new Array(C).fill(10); });
    }

    function morph(img, Bm, mode) {
      var Bref = mode === 'dil' ? Bm.slice().reverse().map(function(r){ return r.slice().reverse(); }) : Bm;
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                if (mode === 'dil' && img[vy][vx] > g[y][x]) g[y][x] = img[vy][vx];
                if (mode === 'ero' && img[vy][vx] < g[y][x]) g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paint(grid, img, cmin, cmax, hue) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var v = img[y][x];
          var t = cmax > cmin ? (v - cmin) / (cmax - cmin) : 0;
          var c = document.createElement('div');
          c.style.cssText = 'width:20px;height:20px;border-radius:3px;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : hue;
          c.style.opacity = v === 0 ? '1' : (0.35 + 0.65 * Math.min(1, t));
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var d = morph(f, B, 'dil');
      var e = morph(f, B, 'ero');
      var ab = morph(e, B, 'dil');
      var fc = morph(d, B, 'ero');

      var grad = f.map(function(r, y){ return r.map(function(_, x){ return d[y][x] - e[y][x]; }); });
      var th   = f.map(function(r, y){ return r.map(function(v, x){ return v - ab[y][x]; }); });
      var bh   = f.map(function(r, y){ return r.map(function(v, x){ return fc[y][x] - v; }); });

      var maxF = Math.max.apply(null, f.map(function(r){ return Math.max.apply(null, r); }));
      var maxG = Math.max.apply(null, grad.map(function(r){ return Math.max.apply(null, r); }));
      var maxTh = Math.max.apply(null, th.map(function(r){ return Math.max.apply(null, r); }));
      var maxBh = Math.max.apply(null, bh.map(function(r){ return Math.max.apply(null, r); }));

      paint(gF, f, 10, maxF || 1, '#7f8c8d');
      paint(gGrad, grad, 0, Math.max(1, maxG), '#8e44ad');
      paint(gTh, th, 0, Math.max(1, maxTh), '#d35400');
      paint(gBh, bh, 0, Math.max(1, maxBh), '#2980b9');
    }

    btnPico.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.min(255, f[y][x] + 60 + Math.floor(Math.random() * 30));
      render();
    });

    btnVale.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.max(0, f[y][x] - 8 - Math.floor(Math.random() * 4));
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMatrix();
      f[2][2] = 80;
      f[6][6] = 2;
      render();
    });

    resetMatrix();
    f[2][2] = 80;
    f[6][6] = 2;
    render();
  }

  function tryInitSimEP0408(){
    var root = document.getElementById('sim-ep0408-gradiente');
    if (root) initSimEP0408(root); else setTimeout(tryInitSimEP0408, 200);
  }
  tryInitSimEP0408();
})();
</script>
</div>
""")

**Figure 4.8:** EP04_08 Simulator: Morphological Gradient, Top-hat and Black-hat


In [ ]:
%%writefile EP04_08.py
# Python code

In [ ]:
TestSuite("EP04_08.py").run()

### EP04_09 🗺️ Distance Transform and the Object's "Core"

In **mobile robotics**, when planning a route within a corridor, the robot wants to know not only *where* free space exists, but also **how far** each free point is from the nearest wall. The safest paths tend to pass through the corridor's "core," away from obstacles.

The **morphological distance transform** assigns to each pixel a value representing its distance to the nearest edge, according to the metric defined by the structuring element. Pixels near the edge receive low values, while more internal pixels receive higher values. The pixel with the maximum value corresponds to the most protected region of the object, often associated with its morphological center.

See [Figure 4.9](#fig-04-sim-ep0409-distancia) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Image dimensions:** read the integers $L$ (rows) and $C$ (columns) of image $f$.
2. **Dimensions of $B$:** read the integers $L_B$ (rows) and $C_B$ (columns) of the structuring element.
3. **Structuring element:** read the matrix $b$, containing value $0$ at the center and negative values at other positions.
4. **Image:** read the binary matrix $f$ (values $0$ or $1$), row by row.
5. **Preparation:** multiply the image by $L\times C$, ensuring that internal pixels have an initial value sufficiently high for distance propagation.
6. **Distance transform:** compute the distance matrix using the method `mm.dist1(f,b)`.
7. **Output:** display the matrix resulting from the distance transform.

#### 📌 Computational Constraints

* Use the weighted erosion implementation provided by the library.
* The structuring element may contain arbitrary negative values.
* The transform must be obtained by iteratively applying weighted erosions until a fixed point is reached.

**⚠️ Crucial Note on Matrix Reading:** Since the structuring element may contain negative integer values (e.g., `-1` and `-99`), **do not use the `mm.readImg` function to read matrix $b$**. This function converts data to `uint8` type, causing *underflow* and corrupting negative values. Read the $L_B$ rows of $b$ manually using the default `int` type. Image $f$ can continue to be read normally using `mm.readImg`.

#### 🧠 Theoretical Foundation

| Concept                              | Meaning                                                                              | Visual Impact                               |
| ------------------------------------ | ------------------------------------------------------------------------------------ | ------------------------------------------- |
| **$\text{dist}(y,x)$**               | Morphological distance to the nearest edge according to the metric defined by $b$    | More internal pixels receive higher values  |
| **Maximum value**                    | Pixel farthest from the edge                                                         | Approximates the object's morphological center |
| **Weighted structuring element**     | Defines the displacement costs between neighboring pixels                            | Determines the distance metric used         |
| **Thin objects**                     | Narrow regions of the object                                                         | Produce low distance values                 |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: integer $L$.
* Line 2: integer $C$.
* Line 3: integer $L_B$.
* Line 4: integer $C_B$.
* Next $L_B$ lines: integer elements of matrix $b$.
* Next $L$ lines: binary elements ($0$ or $1$) of matrix $f$.

⚠️ **Implementation note:** The elements of matrix $f$ (0 or 1) must be multiplied by **255** to generate an adequate binary image ($0$ and $255$) before applying the Distance Transform (DT).

**Output:**

* Distance transform matrix with $L$ rows and $C$ columns.

#### 📌 Example

| Input                                                                                                                                                          | Output                                                                                                | Observation                              |
| ---------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- | ---------------------------------------- |
| 5<br>9<br>3<br>3<br>-99 -1 -99<br>-1 0 -1<br>-99 -1 -99<br>0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | 0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 2 2 2 2 2 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | Result of the distance transform. |

**Note:** the value `-99` acts as a practical approximation of $-\infty$, preventing propagation along diagonals. Thus, only horizontal and vertical neighbors contribute to the distance, producing the Manhattan distance.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0409-distancia" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🗺️ EP04_09 Simulator: Distance Transform</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Erosion Layers</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Click cells to draw your own object or select a predefined shape to calculate the cascading distance map.</p>

    <!-- Grade f Original Clicável -->
    <div style="display:flex;justify-content:center;margin-bottom:14px;">
      <div id="sim_ep0409_grid_f" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

    <!-- Botões de Formas Predefinidas -->
    <div style="text-align:center;margin-bottom:14px;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0409_btn_corredor" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📐 Corridor</button>
      <button id="sim_ep0409_btn_disco" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬤ Disc</button>
      <button id="sim_ep0409_btn_l" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📏 L-Shape</button>
    </div>

    <!-- Título do Mapa de Distâncias -->
    <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Calculated Distance Map</span>

    <!-- Grade de Distâncias -->
    <div style="display:flex;justify-content:center;">
      <div id="sim_ep0409_grid_dist" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0409(root){
    if (!root || root.dataset.simEp0409Init) return;
    root.dataset.simEp0409Init = "1";

    var gF = root.querySelector('#sim_ep0409_grid_f');
    var gD = root.querySelector('#sim_ep0409_grid_dist');

    var L = 5, C = 9, f = [];
    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];

    function setCorredor() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function setDisco() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var cy = 2, cx = 4;
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (Math.pow(y - cy, 2) + Math.pow((x - cx) * 0.6, 2) <= 4) f[y][x] = 1;
        }
      }
    }

    function setL() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 3; x++) f[y][x] = 1;
      }
      for (var y = 2; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function sameMatrix(a, b) {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (a[y][x] !== b[y][x]) return false;
        }
      }
      return true;
    }

    function render() {
      gF.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:32px;height:32px;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = f[yy][xx] ? '#16a085' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            gF.appendChild(c);
          })(y, x);
        }
      }

      var dist = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var atual = f.map(function(r){ return r.slice(); });
      var nivel = 0;

      while (atual.some(function(r){ return r.some(function(v){ return v === 1; }); })) {
        nivel++;
        for (var y = 0; y < L; y++) {
          for (var x = 0; x < C; x++) {
            if (atual[y][x] === 1) dist[y][x] = nivel;
          }
        }
        var prox = erode(atual, B);
        if (sameMatrix(prox, atual)) break;
        atual = prox;
      }

      gD.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = dist[y][x];
          var t = v / (nivel || 1);
          var inten = Math.round(220 - t * 170);

          c.style.cssText = 'width:32px;height:32px;border-radius:6px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : 'rgb(' + (inten - 60) + ',' + inten + ',' + (inten - 30) + ')';
          c.style.color = v > 0 ? '#ffffff' : '#8a8371';
          c.textContent = v || '';
          gD.appendChild(c);
        }
      }
    }

    root.querySelector('#sim_ep0409_btn_corredor').addEventListener('click', function(){ setCorredor(); render(); });
    root.querySelector('#sim_ep0409_btn_disco').addEventListener('click', function(){ setDisco(); render(); });
    root.querySelector('#sim_ep0409_btn_l').addEventListener('click', function(){ setL(); render(); });

    setCorredor();
    render();
  }

  function tryInitSimEP0409(){
    var root = document.getElementById('sim-ep0409-distancia');
    if (root) initSimEP0409(root); else setTimeout(tryInitSimEP0409, 200);
  }
  tryInitSimEP0409();
})();
</script>
</div>
""")

**Figure 4.9:** Simulador EP04_09: Distância por Transformada (Camadas de Erosão)


In [ ]:
%%writefile EP04_09.py
# Python code

In [ ]:
TestSuite("EP04_09.py").run()

### EP04_10 🪙 *Blob* Separation, Labeling, and Descriptors

In a **coin production line**, it is common for pieces to touch one another on the conveyor belt, forming a single connected blob in the image—a naive count would yield the wrong total. The classic solution combines morphological operations and connectivity analysis: first, an **erosion** reduces or breaks fragile connections between objects, and then **connected component labeling** separates each object into a distinct region. Finally, **geometric descriptors** (area and bounding box) summarize each detected component.

See [Figure 4.10](#fig-04-sim-ep0410-rotulacao) for a simulation of this EP.

#### 📋 Implementation Guidelines

1. **Image dimensions:** read the integers $L$ (rows) and $C$ (columns) from $f$.

2. **Dimensions of $B$:** read the integers $L_B$ (rows) and $C_B$ (columns) of the structuring element.

3. **Structuring element:** read the matrix $B$, containing values $0$ or $1$, row by row.

4. **Data:** read the binary matrix $f$ (values $0$ or $1$), row by row.

5. **Separation:** compute
   $$
   f_{ero} = f \ominus B
   $$
   using flat binary erosion (as in EP04_04), eliminating fragile connections between objects.

6. **Labeling:** on $f_{ero}$, identify connected components using connectivity defined by the neighborhood $B$. Labeling must follow *raster* scanning: upon finding an unlabeled pixel with value $1$, assign a new increasing integer label starting from 1 and propagate that label to the entire connected region.

7. **Descriptors:** for each label $k$, compute:

   * **Area:** number of pixels belonging to the label;
   * **Bounding box:** $$(y_{min}, x_{min}, y_{max}, x_{max})$$

8. **Output:** display the total number of labels and then one line per label in the format:
   $$
   k,\ \text{area},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
   $$

#### 📌 Computational Constraints

* Erosion must be applied before labeling.
* Connectivity is fixed and defined by the neighborhood above.
* The structuring element $B$ does not interfere with labeling connectivity.
* No padding in any step.
* The order of labels follows first discovery in *raster* scanning.

#### 🧠 Theoretical Background

| Concept           | Meaning                                    | Impact                                             |
| ----------------- | ------------------------------------------ | -------------------------------------------------- |
| Thin bridge       | Narrow connection between objects          | Can be removed by morphological erosion            |
| Connectivity      | Defined by the set $$\mathcal{N}(y,x)$$    | Determines which pixels belong to the same component |
| Area               | Number of pixels per component             | Direct estimate of object size                     |
| Bounding box       | Spatial extent of the label                | Geometric summary of the component                 |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: integer $L$
* Line 2: integer $C$
* Line 3: integer $L_B$
* Line 4: integer $C_B$
* Next $L_B$ lines: matrix $B$
* Next $L$ lines: matrix $f$

**Output:**

* Line 1: total number of labels found
* Following lines:
  $$
  k,\ \text{area},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
  $$

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0410-rotulacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪙 EP04_10 Simulator: Coins Stuck Together → Separated → Counted</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">erosion + labeling + descriptors</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Adjust the bridge thickness between the coins and observe how morphological erosion separates the objects for counting and descriptor extraction (area and bounding box).</p>

    <!-- Controle de Espessura da Ponte -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#b9770e;">Bridge Thickness Between Coins</label><br>
      <input type="range" id="sim_ep0410_sl_p" min="1" max="3" step="1" value="1" style="width:60%;cursor:pointer;accent-color:#b9770e;margin-top:6px;">
      <span id="sim_ep0410_vl_p" style="font-family:monospace;font-size:12px;font-weight:700;color:#b9770e;margin-left:8px;">1 px</span>
    </div>

    <!-- Comparativo Lado a Lado: f original vs Rótulos Pós-Erosão -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original (ligadas) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original (Connected)</span>
        <div id="sim_ep0410_grid_f" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Após Erosão + Rótulos -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">After Erosion + Labels</span>
        <div id="sim_ep0410_grid_lab" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Painel Informativo / Descritores -->
    <div id="sim_ep0410_info" style="background:#fef5e7;border:1px solid #f8c471;border-radius:8px;padding:10px 14px;font-size:11px;color:#7d5a00;text-align:center;line-height:1.5;"></div>

  </div>
</div>

<script>
(function(){
  function initSimEP0410(root){
    if (!root || root.dataset.simEp0410Init) return;
    root.dataset.simEp0410Init = "1";

    var slP  = root.querySelector('#sim_ep0410_sl_p');
    var vlP  = root.querySelector('#sim_ep0410_vl_p');
    var gF   = root.querySelector('#sim_ep0410_grid_f');
    var gL   = root.querySelector('#sim_ep0410_grid_lab');
    var info = root.querySelector('#sim_ep0410_info');

    var L = 7, C = 10;
    var B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
    var palette = ['#e74c3c', '#27ae60', '#2980b9', '#8e44ad', '#d35400'];

    function buildF(p) {
      var f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 6; y++) {
        for (var x = 1; x < 4; x++) f[y][x] = 1;
      }
      for (var y = 1; y < 6; y++) {
        for (var x = 6; x < 9; x++) f[y][x] = 1;
      }
      var midRow = 3;
      for (var dy = 0; dy < p; dy++) {
        var ry = midRow - Math.floor(p / 2) + dy;
        for (var x = 4; x < 6; x++) f[ry][x] = 1;
      }
      return f;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function labelK8(img) {
      var labels = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var dirs = [[-1, -1], [-1, 0], [-1, 1], [0, -1], [0, 1], [1, -1], [1, 0], [1, 1]];
      var cur = 0, desc = [];

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (img[y][x] === 1 && labels[y][x] === 0) {
            cur++;
            var stack = [[y, x]];
            labels[y][x] = cur;
            var area = 0, miny = y, maxy = y, minx = x, maxx = x;

            while (stack.length) {
              var cell = stack.pop();
              var cy = cell[0], cx = cell[1];
              area++;
              if (cy < miny) miny = cy;
              if (cy > maxy) maxy = cy;
              if (cx < minx) minx = cx;
              if (cx > maxx) maxx = cx;

              dirs.forEach(function(d){
                var ny = cy + d[0], nx = cx + d[1];
                if (ny >= 0 && ny < L && nx >= 0 && nx < C && img[ny][nx] === 1 && labels[ny][nx] === 0) {
                  labels[ny][nx] = cur;
                  stack.push([ny, nx]);
                }
              });
            }
            desc.push({k: cur, area: area, miny: miny, minx: minx, maxy: maxy, maxx: maxx});
          }
        }
      }
      return {labels: labels, desc: desc};
    }

    function paintBin(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] ? '#b9770e' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintLabels(grid, labels) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var k = labels[y][x];
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;color:#ffffff;box-sizing:border-box;';
          c.style.background = k > 0 ? palette[(k - 1) % palette.length] : '#fafaf7';
          c.textContent = k > 0 ? k : '';
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var p = parseInt(slP.value, 10) || 1;
      vlP.textContent = p + ' px';
      var f = buildF(p);
      var fe = erode(f, B);
      var res = labelK8(fe);

      paintBin(gF, f);
      paintLabels(gL, res.labels);

      var txt = '<b>' + res.desc.length + ' objeto(s) detectado(s) após a erosão.</b><br>';
      res.desc.forEach(function(d){
        txt += 'Rótulo ' + d.k + ': área = ' + d.area + ', bbox = (' + d.miny + ',' + d.minx + ') → (' + d.maxy + ',' + d.maxx + ')<br>';
      });
      if (res.desc.length < 2) {
        txt += '<i>A ponte ainda é espessa demais para a erosão 3×3 — as moedas continuam fundidas em 1 só objeto.</i>';
      }
      info.innerHTML = txt;
    }

    slP.addEventListener('input', render);

    render();
  }

  function tryInitSimEP0410(){
    var root = document.getElementById('sim-ep0410-rotulacao');
    if (root) initSimEP0410(root); else setTimeout(tryInitSimEP0410, 200);
  }
  tryInitSimEP0410();
})();
</script>
</div>
""")

**Figure 4.10:** Simulador EP04_10: Blob Separation, Labeling, and Descriptors


In [ ]:
%%writefile EP04_10.py
# Python code

In [ ]:
TestSuite("EP04_10.py").run()